# RecipeGPT v2 — Test Suite
Run this after training to evaluate model quality across multiple dimensions.


In [ ]:
# ── Load model (run after training notebook) ──────────────────────────────
import torch, torch.nn as nn, math
from torch.nn import functional as F

# Paste your hyperparams here (same as training)
n_embd=512; n_head=8; n_layer=8; block_size=384; dropout=0.0  # dropout=0 at inference
device = 'cuda' if torch.cuda.is_available() else 'cpu'

ckpt = torch.load('recipe_gpt_v2.pt', map_location=device)
vocab_size = ckpt['vocab_size']
stoi = ckpt['stoi']
itos = ckpt['itos']
encode = lambda s: [stoi[c] for c in s if c in stoi]
decode = lambda l: ''.join([itos[i] for i in l])
print(f'Model loaded. Vocab size: {vocab_size}')


In [ ]:
# ── Re-define model architecture ───────────────────────────────────────────
class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.c_attn  = nn.Linear(n_embd, 3*n_embd, bias=False)
        self.c_proj  = nn.Linear(n_embd, n_embd,   bias=False)
        self.attn_drop  = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.register_buffer('bias', torch.tril(torch.ones(block_size, block_size)).view(1,1,block_size,block_size))
    def forward(self, x):
        B,T,C = x.size()
        q,k,v = self.c_attn(x).split(n_embd, dim=2)
        hs = C//n_head
        k = k.view(B,T,n_head,hs).transpose(1,2)
        q = q.view(B,T,n_head,hs).transpose(1,2)
        v = v.view(B,T,n_head,hs).transpose(1,2)
        att = (q @ k.transpose(-2,-1)) * (1.0/math.sqrt(hs))
        att = att.masked_fill(self.bias[:,:,:T,:T]==0, float('-inf'))
        att = self.attn_drop(F.softmax(att, dim=-1))
        return self.resid_drop(self.c_proj((att@v).transpose(1,2).contiguous().view(B,T,C)))

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.c_fc   = nn.Linear(n_embd, 4*n_embd, bias=False)
        self.c_proj = nn.Linear(4*n_embd, n_embd, bias=False)
        self.drop   = nn.Dropout(dropout)
    def forward(self, x): return self.drop(self.c_proj(F.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln_1=nn.LayerNorm(n_embd); self.attn=CausalSelfAttention()
        self.ln_2=nn.LayerNorm(n_embd); self.mlp=MLP()
    def forward(self,x): return x + self.mlp(self.ln_2(x + self.attn(self.ln_1(x))))

class RecipeGPTv2(nn.Module):
    def __init__(self):
        super().__init__()
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(vocab_size,n_embd), wpe=nn.Embedding(block_size,n_embd),
            drop=nn.Dropout(dropout), h=nn.ModuleList([Block() for _ in range(n_layer)]),
            ln_f=nn.LayerNorm(n_embd)))
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
    def forward(self, idx, targets=None):
        B,T = idx.shape
        x = self.transformer.drop(self.transformer.wte(idx)+self.transformer.wpe(torch.arange(T,device=idx.device)))
        for block in self.transformer.h: x = block(x)
        logits = self.lm_head(self.transformer.ln_f(x))
        loss = F.cross_entropy(logits.view(B*T,-1), targets.view(B*T)) if targets is not None else None
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
        for _ in range(max_new_tokens):
            logits,_ = self(idx[:,-block_size:])
            logits = logits[:,-1,:] / temperature
            if top_k:
                v,_ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:,[-1]]] = float('-inf')
            if top_p:
                sl,si = torch.sort(logits, descending=True)
                cp = torch.cumsum(F.softmax(sl,dim=-1),dim=-1)
                sl[cp - F.softmax(sl,dim=-1) > top_p] = float('-inf')
                logits = torch.zeros_like(logits).scatter_(1,si,sl)
            idx = torch.cat((idx, torch.multinomial(F.softmax(logits,dim=-1),1)), dim=1)
        return idx

model = RecipeGPTv2().to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Ready. Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')


## Test 1 — Structured Recipe Generation

In [ ]:
def gen(prompt, temp=0.8, top_k=50, top_p=0.92, tokens=600):
    ctx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    out = model.generate(ctx, tokens, temperature=temp, top_k=top_k, top_p=top_p)
    return decode(out[0].tolist())

dishes = ['Paneer Tikka', 'Mutton Curry', 'Vegetable Biryani', 'Kheer', 'Aloo Paratha']
for dish in dishes:
    print('='*65)
    print(f'PROMPT: {dish}')
    print('='*65)
    print(gen(f'[RECIPE]\nNAME: {dish}', temp=0.75, top_k=40, top_p=0.92, tokens=550))
    print()


## Test 2 — Quality Metrics

In [ ]:
# Checks how well the model has learned recipe structure
import re

def quality_score(text):
    scores = {}
    scores['has_ingredients'] = 'INGREDIENTS' in text
    scores['has_method']      = 'METHOD' in text
    scores['has_tips']        = 'TIPS' in text
    scores['has_serves']      = 'SERVES' in text
    scores['has_time']        = 'TIME' in text
    scores['has_quantities']  = bool(re.search(r'\d+\s*(g|kg|tbsp|tsp|cup|ml|liter)', text))
    scores['has_numbered_steps'] = bool(re.search(r'\d+\.\s+[A-Z]', text))
    scores['has_real_spices'] = any(s in text.lower() for s in
        ['turmeric','garam masala','cumin','coriander','cardamom','saffron','ghee','chili'])
    scores['no_gibberish']    = not bool(re.search(r'[a-z]{15,}', text))  # no 15+ char runs
    total = sum(scores.values())
    pct   = total / len(scores) * 100
    return scores, total, pct

print('QUALITY EVALUATION ACROSS 5 DISHES')
print('='*65)
all_scores = []
for dish in ['Paneer', 'Chicken Curry', 'Dal Tadka', 'Halwa', 'Samosa']:
    text = gen(f'[RECIPE]\nNAME: {dish}', temp=0.75, tokens=500)
    scores, total, pct = quality_score(text)
    all_scores.append(pct)
    status = {True:'✓', False:'✗'}
    checks = '  '.join(f"{status[v]} {k.replace('has_','').replace('_',' ')}" for k,v in scores.items())
    print(f'\n{dish}: {pct:.0f}%')
    print(f'  {checks}')

print(f'\n{"="*65}')
print(f'AVERAGE QUALITY SCORE: {sum(all_scores)/len(all_scores):.1f}%')


## Test 3 — Temperature Comparison

In [ ]:
# Same dish at 3 temperatures to show the difference
print('TEMPERATURE COMPARISON — Dal Makhani')
for temp, label in [(0.6,'Conservative'),(0.9,'Balanced'),(1.3,'Creative')]:
    print(f'\n{"-"*65}')
    print(f'Temperature {temp} — {label}')
    print('-'*65)
    print(gen('[RECIPE]\nNAME: Dal Makhani', temp=temp, top_k=50, top_p=0.92, tokens=400))


## Test 4 — Perplexity (Lower = Better)

In [ ]:
# Perplexity measures how surprised the model is by new text
# Lower perplexity = model has learned the pattern well
# Good target: < 5.0 for a well-trained small model on this corpus

@torch.no_grad()
def compute_perplexity(text_sample, max_len=500):
    ids = encode(text_sample[:max_len])
    if len(ids) < 10: return float('inf')
    x = torch.tensor([ids[:-1]], dtype=torch.long, device=device)
    y = torch.tensor([ids[1:]],  dtype=torch.long, device=device)
    if x.shape[1] > block_size: x,y = x[:,:block_size], y[:,:block_size]
    _, loss = model(x, y)
    return torch.exp(loss).item()

test_texts = [
    ('Training-style recipe', '[RECIPE]\nNAME: Butter Chicken\nCATEGORY: Main Course\nINGREDIENTS:\n- 500g chicken'),
    ('Random English text',   'The weather today is quite pleasant and the birds are singing in the trees outside'),
    ('Random chars',          'xkqzwvmplrfbdgjhstyuioeanc xkqzwvmplrfbdgjh'),
]

print('PERPLEXITY TEST (lower = model knows this pattern better)')
print('='*55)
for label, txt in test_texts:
    ppl = compute_perplexity(txt)
    bar = '█' * min(int(ppl), 40)
    print(f'{label:<30} PPL: {ppl:>7.2f}  {bar}')
print('\nIdeal: recipe text << random English << random chars')
